## 1. Carregamento e Visualização dos Dados

In [1]:
import pandas as pd
import numpy as np
from itertools import product
from collections import defaultdict

URL = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv"
df_raw = pd.read_csv(URL)

print(f"Shape: {df_raw.shape}")
df_raw.head(3)

Shape: (186, 11)


,Carimbo de data/hora,Você ficou gripado no ano passado ?,Você tomou vacina da gripe no ano passado?,"Você frequentou no ano passado, semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)",Você viajou no ano passado mais de 100 km de distância?,"Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?",Quantas horas você dormiu em média por noite no ano passado?,Você praticou atividade física no ano passado?,Você se alimentou de forma balanceada no ano passado?,"Em média, quantas vezes você lavou as mãos por dia no ano passado?","Na sua percepção, o seu nível de estresse no ano passado foi:"
0,24/03/2026 15:01:35,Sim,Sim,Sim,Poucas vezes por ano,Médio,4 horas ou menos,Sim,Às vezes,3 a 5 vezes,5.0
1,24/03/2026 15:04:20,Sim,Sim,Sim,Nuca,Não,entre 4 e 6 horas,Não,"Não, raramente",Mais de 10 vezes,3.0
2,24/03/2026 15:04:20,Sim,Não,Sim,Poucas vezes por ano,Pouco,mais de 6 horas,Sim,Às vezes,6 a 10 vezes,3.0


In [2]:
# Renomear colunas para nomes curtos
col_map = {
    df_raw.columns[0]: 'timestamp',
    df_raw.columns[1]: 'gripado',
    df_raw.columns[2]: 'vacina',
    df_raw.columns[3]: 'ambientes_cheios',
    df_raw.columns[4]: 'viagem',
    df_raw.columns[5]: 'alergia',
    df_raw.columns[6]: 'horas_sono',
    df_raw.columns[7]: 'atividade_fisica',
    df_raw.columns[8]: 'alimentacao',
    df_raw.columns[9]: 'lavagem_maos',
    df_raw.columns[10]: 'estresse',
}

df = df_raw.rename(columns=col_map).drop(columns=['timestamp'])
print("Colunas:", df.columns.tolist())
print(f"\nTotal de instâncias: {len(df)}")
df.head()

Colunas: ['gripado', 'vacina', 'ambientes_cheios', 'viagem', 'alergia', 'horas_sono', 'atividade_fisica', 'alimentacao', 'lavagem_maos', 'estresse']

Total de instâncias: 186


,gripado,vacina,ambientes_cheios,viagem,alergia,horas_sono,atividade_fisica,alimentacao,lavagem_maos,estresse
0,Sim,Sim,Sim,Poucas vezes por ano,Médio,4 horas ou menos,Sim,Às vezes,3 a 5 vezes,5.0
1,Sim,Sim,Sim,Nuca,Não,entre 4 e 6 horas,Não,"Não, raramente",Mais de 10 vezes,3.0
2,Sim,Não,Sim,Poucas vezes por ano,Pouco,mais de 6 horas,Sim,Às vezes,6 a 10 vezes,3.0
3,Sim,Não,Não,Nuca,Muito,mais de 6 horas,Sim,Às vezes,2 vezes ou menos,2.0
4,Sim,Sim,Sim,Pelo menos uma vez por mês,Médio,entre 4 e 6 horas,Não,Às vezes,6 a 10 vezes,4.0


In [3]:
# Distribuição da classe alvo
print("Distribuição da classe alvo 'gripado':")
print(df['gripado'].value_counts())
print()
print("Distribuição em percentual:")
print(df['gripado'].value_counts(normalize=True).map('{:.1%}'.format))

Distribuição da classe alvo 'gripado':
gripado
Sim    109
Não     77
Name: count, dtype: int64

Distribuição em percentual:
gripado
Sim    58.6%
Não    41.4%
Name: proportion, dtype: object


## 2. Pré-processamento

Antes de rodar o PRISM, vamos:
1. Remover linhas com valores ausentes
2. Garantir que todos os atributos sejam categóricos (strings)
3. Separar features da classe alvo

In [4]:
df_clean = df.dropna().copy()

# Garantir tipos string em todos os atributos
for col in df_clean.columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Normalizar atributo estresse: remover ".0" para deixar como inteiro legível
df_clean["estresse"] = df_clean["estresse"].str.replace(r"\.0$", "", regex=True)

print(f"Instâncias após limpeza: {len(df_clean)}")
print()

# Verificar valores únicos por atributo
for col in df_clean.columns:
    vals = df_clean[col].unique()
    print(f"  {col}: {sorted(vals)}")

Instâncias após limpeza: 185

  gripado: ['Não', 'Sim']
  vacina: ['Não', 'Sim']
  ambientes_cheios: ['Não', 'Sim']
  viagem: ['Nuca', 'Pelo menos uma vez por mês', 'Poucas vezes por ano']
  alergia: ['Muito', 'Médio', 'Não', 'Pouco']
  horas_sono: ['4 horas ou menos', 'entre 4 e 6 horas', 'mais de 6 horas']
  atividade_fisica: ['Não', 'Sim']
  alimentacao: ['Não, raramente', 'Sim, a maior parte do tempo', 'Às vezes']
  lavagem_maos: ['2 vezes ou menos', '3 a 5 vezes', '6 a 10 vezes', 'Mais de 10 vezes']
  estresse: ['1', '2', '3', '4', '5']


In [5]:
# Separar features e classe
features = [c for c in df_clean.columns if c != 'gripado']
target = 'gripado'
classes = df_clean[target].unique().tolist()

print(f"Atributos ({len(features)}): {features}")
print(f"Classes: {classes}")

Atributos (9): ['vacina', 'ambientes_cheios', 'viagem', 'alergia', 'horas_sono', 'atividade_fisica', 'alimentacao', 'lavagem_maos', 'estresse']
Classes: ['Sim', 'Não']


## 3. Implementação do Algoritmo PRISM

### Lógica principal

Para cada **classe alvo** $c$:
1. Comece com antecedente vazio e conjunto completo de dados
2. Para cada par (atributo, valor), calcule $p(c \mid atributo = valor)$
3. Escolha o par com maior probabilidade (desempate: maior cobertura positiva)
4. Adicione esse par ao antecedente e restrinja o subconjunto
5. Repita até que a regra seja **perfeita** ($p = 1.0$) ou não haja mais atributos
6. Remova os exemplos cobertos e repita para gerar a próxima regra
7. Pare quando não houver mais exemplos da classe

In [6]:
def prism_rule(subset, target_class, target_col, features):
    """
    Gera UMA regra para target_class usando o algoritmo PRISM.
    Retorna (antecedente, exemplos_cobertos) ou (None, None) se não possível.
    """
    current = subset.copy()
    antecedent = []
    used_attrs = set()

    while True:
        # Contar positivos no subconjunto atual
        n_pos = (current[target_col] == target_class).sum()
        n_tot = len(current)

        if n_tot == 0 or n_pos == 0:
            return None, None

        # Se p = 1.0, regra perfeita
        if n_pos == n_tot:
            covered = current[current[target_col] == target_class]
            return antecedent, covered

        # Calcular p(class | attr = val) para todos os pares disponíveis
        best_p = -1
        best_cov = -1
        best_pair = None

        available = [f for f in features if f not in used_attrs]
        if not available:
            # Sem mais atributos: retorna regra impura (melhor esforço)
            covered = current[current[target_col] == target_class]
            return antecedent, covered

        for attr in available:
            for val in current[attr].unique():
                sub = current[current[attr] == val]
                n_sub_pos = (sub[target_col] == target_class).sum()
                n_sub_tot = len(sub)
                if n_sub_tot == 0:
                    continue
                p = n_sub_pos / n_sub_tot
                cov = n_sub_pos
                if (p > best_p) or (p == best_p and cov > best_cov):
                    best_p = p
                    best_cov = cov
                    best_pair = (attr, val)

        if best_pair is None:
            return None, None

        attr, val = best_pair
        antecedent.append((attr, val))
        used_attrs.add(attr)
        current = current[current[attr] == val]


def prism(df, target_col, features):
    """
    Executa o algoritmo PRISM completo para todas as classes.
    Retorna dicionário: classe -> lista de antecedentes (lista de pares).
    """
    classes = df[target_col].unique().tolist()
    all_rules = {}

    for cls in classes:
        rules = []
        remaining = df.copy()

        while True:
            # Verificar se ainda há exemplos da classe
            if (remaining[target_col] == cls).sum() == 0:
                break

            antecedent, covered = prism_rule(remaining, cls, target_col, features)

            if antecedent is None or covered is None or len(covered) == 0:
                break

            rules.append(antecedent)

            # Remover exemplos cobertos
            remaining = remaining.drop(index=covered.index)

        all_rules[cls] = rules

    return all_rules

print("Funções do algoritmo PRISM definidas.")

Funções do algoritmo PRISM definidas.


## 4. Executando o PRISM no Dataset de Gripe

In [7]:
rules = prism(df_clean, target, features)

print("=" * 60)
print("REGRAS GERADAS PELO ALGORITMO PRISM")
print("=" * 60)

total_rules = 0
for cls, cls_rules in rules.items():
    print(f"\n>>> Classe alvo: gripado = '{cls}'")
    if not cls_rules:
        print("  (nenhuma regra gerada)")
    for i, antecedent in enumerate(cls_rules, 1):
        cond = " AND ".join([f"{a} = {v}" for a, v in antecedent])
        print(f"  R{i}: IF {cond} THEN gripado = {cls}")
        total_rules += 1

print(f"\nTotal de regras geradas: {total_rules}")

REGRAS GERADAS PELO ALGORITMO PRISM

>>> Classe alvo: gripado = 'Sim'
  R1: IF horas_sono = 4 horas ou menos THEN gripado = Sim
  R2: IF viagem = Nuca AND alergia = Não THEN gripado = Sim
  R3: IF alergia = Muito AND ambientes_cheios = Não THEN gripado = Sim
  R4: IF alergia = Muito AND alimentacao = Não, raramente THEN gripado = Sim
  R5: IF alergia = Médio AND estresse = 3 THEN gripado = Sim
  R6: IF estresse = 5 AND alergia = Muito AND viagem = Poucas vezes por ano THEN gripado = Sim
  R7: IF lavagem_maos = 6 a 10 vezes AND alergia = Médio AND estresse = 4 THEN gripado = Sim
  R8: IF estresse = 5 AND lavagem_maos = 2 vezes ou menos AND atividade_fisica = Sim THEN gripado = Sim
  R9: IF viagem = Poucas vezes por ano AND estresse = 5 AND lavagem_maos = Mais de 10 vezes THEN gripado = Sim
  R10: IF viagem = Poucas vezes por ano AND estresse = 5 AND ambientes_cheios = Não THEN gripado = Sim
  R11: IF viagem = Poucas vezes por ano AND estresse = 5 AND alimentacao = Sim, a maior parte do 

## 5. Avaliação das Regras

Vamos avaliar cada regra gerada calculando:
- **Cobertura**: quantas instâncias são cobertas pelo antecedente
- **Acurácia da regra**: proporção de instâncias cobertas que têm a classe correta
- **Cobertura positiva**: instâncias cobertas com a classe alvo correta

In [9]:
def apply_rule(df, antecedent):
    """Retorna máscara booleana das instâncias cobertas pela regra."""
    mask = pd.Series([True] * len(df), index=df.index)
    for attr, val in antecedent:
        mask &= (df[attr] == val)
    return mask

print(f"{'Classe':<8} {'#':<4} {'Regra (resumida)':<65} {'Cob':>5} {'Prec':>7}")
print("-" * 95)

for cls, cls_rules in rules.items():
    for i, antecedent in enumerate(cls_rules, 1):
        mask = apply_rule(df_clean, antecedent)
        covered = mask.sum()
        if covered == 0:
            prec = 0.0
        else:
            prec = (df_clean.loc[mask, target] == cls).sum() / covered
        cond = " AND ".join([f"{a}={v}" for a, v in antecedent])
        cond_short = cond[:62] + "..." if len(cond) > 65 else cond
        print(f"{cls:<8} R{i:<3} {cond_short:<65} {covered:>5} {prec:>7.2%}")

Classe   #    Regra (resumida)                                                    Cob    Prec
-----------------------------------------------------------------------------------------------
Sim      R1   horas_sono=4 horas ou menos                                           4 100.00%
Sim      R2   viagem=Nuca AND alergia=Não                                           6 100.00%
Sim      R3   alergia=Muito AND ambientes_cheios=Não                                2 100.00%
Sim      R4   alergia=Muito AND alimentacao=Não, raramente                          2 100.00%
Sim      R5   alergia=Médio AND estresse=3                                          6 100.00%
Sim      R6   estresse=5 AND alergia=Muito AND viagem=Poucas vezes por ano          6 100.00%
Sim      R7   lavagem_maos=6 a 10 vezes AND alergia=Médio AND estresse=4           10 100.00%
Sim      R8   estresse=5 AND lavagem_maos=2 vezes ou menos AND atividade_fis...     5 100.00%
Sim      R9   viagem=Poucas vezes por ano AND estresse=5 A

## 6. Classificação com as Regras Geradas

Aplicamos as regras ao dataset completo para prever a classe. Uma instância é classificada pela **primeira regra** que a cobre (ordenadas por classe e índice).  
Se nenhuma regra cobrir a instância, a classe mais frequente (default) é atribuída.

In [10]:
def predict_prism(df, rules, target_col, default_class=None):
    """Aplica regras PRISM e retorna série de predições."""
    if default_class is None:
        default_class = df[target_col].value_counts().idxmax()

    predictions = pd.Series([default_class] * len(df), index=df.index)

    # Aplicar regras em ordem inversa para que as primeiras regras tenham prioridade
    for cls, cls_rules in rules.items():
        for antecedent in reversed(cls_rules):
            mask = apply_rule(df, antecedent)
            predictions[mask] = cls

    return predictions

y_pred = predict_prism(df_clean, rules, target)
y_true = df_clean[target]

accuracy = (y_pred == y_true).mean()
print(f"Acurácia geral no conjunto de treino: {accuracy:.2%}")
print(f"Instâncias totais: {len(y_true)}")
print(f"Acertos: {(y_pred == y_true).sum()}")
print(f"Erros: {(y_pred != y_true).sum()}")

Acurácia geral no conjunto de treino: 95.68%
Instâncias totais: 185
Acertos: 177
Erros: 8
